# Run SCEPTER Scenarios

This notebook is the execution step after `04_test_scepter_inputs.ipynb`. It reads the staged SCEPTER run table built from ESA cropland raster blocks, HWSD2 soil-map labels/defaults, and CHIRPS rainfall inputs. It writes one run folder/config file per model-unit/scenario combination and can optionally call an external SCEPTER command.

By default this notebook stages configs and a manifest only. Set `ERW_RUN_EXTERNAL_SCEPTER=true` when the local SCEPTER executable is ready and you want this notebook to execute the selected batch.


## Workflow

1. **Load staged run table** from `data/scepter_runs/inputs/scepter_runs.csv`.
2. **Validate notebook 04 provenance**: ESA cropland source, HWSD2 soil map/defaults, and CHIRPS rainfall metadata.
3. **Select runs** with `SCEPTER_RUN_MODE`: smoke, pilot, or final.
4. **Write run configs** under `data/scepter_runs/runs/{run_id}/scepter_config.json`.
5. **Create output folders** under `data/scepter_runs/outputs/{run_id}/`.
6. **Optionally execute SCEPTER** using a command template that receives `{config_path}`, `{run_dir}`, `{output_dir}`, and `{run_id}`.
7. **Write an execution manifest** to `data/scepter_runs/outputs/scepter_execution_manifest.csv`.

The execution manifest is the handoff to notebook `06`, which parses completed SCEPTER outputs and joins them back to raster-derived model-unit geometries.


In [1]:
from pathlib import Path
import os
import shutil
import sys

import pandas as pd

def mount_google_drive_if_colab() -> None:
    try:
        from google.colab import drive
    except ModuleNotFoundError:
        return

    drive.mount("/content/drive")


mount_google_drive_if_colab()

COLAB_PROJECT_ROOT = Path("/content/drive/MyDrive/erw_spatial_mrv")
COLAB_DATA_ROOT = COLAB_PROJECT_ROOT / "data"
LOCAL_PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()


def has_erw_package(project_root: Path) -> bool:
    return (project_root / "src" / "erw_mrv" / "__init__.py").exists()


def source_root_candidates() -> list[Path]:
    cwd = Path.cwd().resolve()
    candidates = [LOCAL_PROJECT_ROOT, COLAB_PROJECT_ROOT]
    for base in (cwd, *cwd.parents):
        candidates.extend((base, base / "erw_spatial_mrv"))
    candidates.extend(
        Path(path)
        for path in (
            "/content/erw_spatial_mrv",
            "/content/enhanced_rock_weathering/erw_spatial_mrv",
            "/content/drive/MyDrive/erw_spatial_mrv",
        )
    )
    unique = []
    for candidate in candidates:
        if candidate not in unique:
            unique.append(candidate)
    return unique


def find_source_project_root() -> Path:
    for candidate in source_root_candidates():
        if has_erw_package(candidate):
            return candidate
    checked = chr(10).join(f"- {candidate}" for candidate in source_root_candidates())
    raise ModuleNotFoundError(
        "Could not find src/erw_mrv. The data can live in Google Drive, but "
        "the notebook still needs the project source folder containing src/erw_mrv. "
        "In Colab, upload/sync the full erw_spatial_mrv project or run from a "
        "checkout that includes src/. Checked: "
        f"{checked}"
    )


SOURCE_PROJECT_ROOT = find_source_project_root()
PROJECT_ROOT = COLAB_PROJECT_ROOT if COLAB_DATA_ROOT.exists() else SOURCE_PROJECT_ROOT
DATA_ROOT = COLAB_DATA_ROOT if COLAB_DATA_ROOT.exists() else PROJECT_ROOT / "data"
os.environ["ERW_MRV_DATA_ROOT"] = str(DATA_ROOT)

SRC = SOURCE_PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"SOURCE_PROJECT_ROOT = {SOURCE_PROJECT_ROOT}")
print(f"DATA_ROOT = {DATA_ROOT}")

from erw_mrv.paths import SCEPTER_INPUTS, SCEPTER_OUTPUTS, SCEPTER_RUN_DIR, ensure_dir
from erw_mrv.scepter import (
    execute_scepter_runs,
    load_scepter_runs,
    select_scepter_runs,
    write_run_configs,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 160)


PROJECT_ROOT = /Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv
SOURCE_PROJECT_ROOT = /Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv
DATA_ROOT = /Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data


## Settings

Use `SCEPTER_RUN_MODE` to choose how much work notebook `05` should stage or execute:

- `"smoke"`: one very short baseline run, used only to prove the binary and adapter work.
- `"pilot"`: four runs for the first model unit, baseline plus all ERW scenarios, so notebook `06` can compute additionality before the expensive final batch.
- `"final"`: all staged model-unit/scenario runs, using production years through the adapter.

SCEPTER is expected to be built locally, outside this notebook. Set `ERW_RUN_EXTERNAL_SCEPTER=true` only after `SCEPTER_EXECUTABLE` points to a working local binary.


In [2]:
# Rebuild setup if this cell is run after a kernel restart without the setup cell.
if "SCEPTER_INPUTS" not in globals():
    from pathlib import Path
    import os
    import shutil
    import sys

    import pandas as pd

    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except ModuleNotFoundError:
        pass

    colab_project_root = Path("/content/drive/MyDrive/erw_spatial_mrv")
    cwd = Path.cwd().resolve()
    candidates = [colab_project_root, cwd, cwd.parent, cwd / "erw_spatial_mrv", cwd.parent / "erw_spatial_mrv"]
    source_project_root = next(
        (candidate for candidate in candidates if (candidate / "src" / "erw_mrv" / "__init__.py").exists()),
        None,
    )
    if source_project_root is None:
        checked = chr(10).join(f"- {candidate}" for candidate in candidates)
        raise ModuleNotFoundError(
            "Could not rebuild notebook setup because src/erw_mrv was not found. "
            "Run the first setup/import cell or sync the full project to /content/drive/MyDrive/erw_spatial_mrv. "
            f"Checked:{chr(10)}{checked}"
        )

    PROJECT_ROOT = colab_project_root if (colab_project_root / "data").exists() else source_project_root
    data_root = PROJECT_ROOT / "data"
    os.environ["ERW_MRV_DATA_ROOT"] = str(data_root)
    src = source_project_root / "src"
    if str(src) not in sys.path:
        sys.path.insert(0, str(src))

    from erw_mrv.paths import SCEPTER_INPUTS, SCEPTER_OUTPUTS, SCEPTER_RUN_DIR, ensure_dir
    from erw_mrv.scepter import execute_scepter_runs, load_scepter_runs, select_scepter_runs, write_run_configs

RUN_TABLE_PATH = SCEPTER_INPUTS / "scepter_runs.csv"
MODEL_UNITS_TABLE_PATH = SCEPTER_INPUTS / "scepter_model_units.csv"
RUN_ROOT = ensure_dir(SCEPTER_RUN_DIR)
OUTPUT_ROOT = ensure_dir(SCEPTER_OUTPUTS)

SCEPTER_RUN_MODE = os.environ.get("ERW_SCEPTER_RUN_MODE", "pilot").lower()
VALID_SCEPTER_RUN_MODES = {"smoke", "pilot", "final"}
if SCEPTER_RUN_MODE not in VALID_SCEPTER_RUN_MODES:
    raise ValueError(f"SCEPTER_RUN_MODE must be one of {sorted(VALID_SCEPTER_RUN_MODES)}, got {SCEPTER_RUN_MODE!r}")

if SCEPTER_RUN_MODE == "smoke":
    MAX_TEST_RUNS = 1
    SCENARIO_IDS = None
    SCEPTER_ADAPTER_EXTRA_ARGS = ""
elif SCEPTER_RUN_MODE == "pilot":
    MAX_TEST_RUNS = 4
    SCENARIO_IDS = None
    SCEPTER_ADAPTER_EXTRA_ARGS = ""
else:
    MAX_TEST_RUNS = None
    SCENARIO_IDS = None
    production_years = os.environ.get("ERW_SCEPTER_PRODUCTION_YEARS", "10")
    SCEPTER_ADAPTER_EXTRA_ARGS = f"--production-years {production_years}"

REQUIRE_REAL_STAGED_INPUTS = True
RUN_EXTERNAL_SCEPTER = os.environ.get("ERW_RUN_EXTERNAL_SCEPTER", "false").lower() in {"1", "true", "yes", "y"}
SCEPTER_SOURCE_ROOT = PROJECT_ROOT / "external" / "SCEPTER"
SCEPTER_EXECUTABLE = Path(os.environ.get("SCEPTER_EXECUTABLE", SCEPTER_SOURCE_ROOT / "scepter")).expanduser()
TIMEOUT_SECONDS = int(os.environ.get("ERW_SCEPTER_TIMEOUT_SECONDS", str(6 * 60 * 60)))

SCEPTER_ADAPTER = PROJECT_ROOT / "scripts" / "run_scepter_adapter.py"

# Build SCEPTER locally outside this notebook, then let the adapter translate JSON configs
# into SCEPTER's native input files before executing the compiled binary.
SCEPTER_COMMAND_TEMPLATE = (
    f"{sys.executable} {SCEPTER_ADAPTER} "
    f"--scepter-root {SCEPTER_SOURCE_ROOT} "
    f"--executable {SCEPTER_EXECUTABLE} "
    "--config {config_path} --output-dir {output_dir} "
    f"--timeout-seconds {TIMEOUT_SECONDS} "
    f"{SCEPTER_ADAPTER_EXTRA_ARGS}"
).strip()

MANIFEST_PATH = OUTPUT_ROOT / "scepter_execution_manifest.csv"
print(f"SCEPTER_RUN_MODE = {SCEPTER_RUN_MODE}")
print(f"MAX_TEST_RUNS = {MAX_TEST_RUNS}")
print(f"SCENARIO_IDS = {SCENARIO_IDS}")
print(f"TIMEOUT_SECONDS = {TIMEOUT_SECONDS}")
print(f"SCEPTER_ADAPTER_EXTRA_ARGS = {SCEPTER_ADAPTER_EXTRA_ARGS or '(smoke-test defaults)'}")
print(f"RUN_EXTERNAL_SCEPTER = {RUN_EXTERNAL_SCEPTER}")
RUN_ROOT, OUTPUT_ROOT, MANIFEST_PATH


SCEPTER_RUN_MODE = final
MAX_TEST_RUNS = None
SCENARIO_IDS = None
TIMEOUT_SECONDS = 21600
SCEPTER_ADAPTER_EXTRA_ARGS = --production-years 1
RUN_EXTERNAL_SCEPTER = False


(PosixPath('/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/runs'),
 PosixPath('/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/outputs'),
 PosixPath('/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/outputs/scepter_execution_manifest.csv'))

## Load Run Table

This table should come from notebook `04`. Each row is one ESA cropland raster-block model unit under one ERW scenario, with cropland, soil-map, soil-default, rainfall, and runoff provenance carried forward.


In [3]:
runs = load_scepter_runs(RUN_TABLE_PATH)

required_notebook04_columns = [
    "cropland_source",
    "cropland_source_path",
    "soil_map_texture_group",
    "soil_map_source_path",
    "soil_source",
    "rainfall_source",
    "rainfall_months_used",
    "runoff_note",
]
optional_notebook04_columns = [
    "cropland_pixels",
    "cropland_pixel_area_m2",
    "cropland_area_note",
    "soil_map_hwsd2_unit_id",
    "soil_map_wrb4",
    "soil_map_fao90",
    "soil_map_clay_pct",
    "soil_map_sand_pct",
    "soil_map_silt_pct",
    "soil_map_join",
    "soil_note",
    "rainfall_source_path",
    "missing_requested_months",
]
notebook04_metadata_columns = required_notebook04_columns + optional_notebook04_columns


def enrich_runs_with_model_unit_metadata(runs_df: pd.DataFrame) -> pd.DataFrame:
    missing_columns = [column for column in notebook04_metadata_columns if column not in runs_df.columns]
    if not missing_columns:
        return runs_df

    if not MODEL_UNITS_TABLE_PATH.exists():
        return runs_df

    units = pd.read_csv(MODEL_UNITS_TABLE_PATH)
    available_metadata = [column for column in missing_columns if column in units.columns]
    if not available_metadata:
        return runs_df

    print(
        "Run table is missing notebook 04 metadata; enriching from "
        f"{MODEL_UNITS_TABLE_PATH}: {available_metadata}"
    )
    return runs_df.merge(units[["model_unit_id", *available_metadata]], on="model_unit_id", how="left")


runs = enrich_runs_with_model_unit_metadata(runs)


def fill_missing_soil_map_labels(runs_df: pd.DataFrame) -> pd.DataFrame:
    runs_df = runs_df.copy()
    if "soil_map_texture_group" not in runs_df.columns:
        return runs_df

    missing_soil_label = runs_df["soil_map_texture_group"].isna()
    if not missing_soil_label.any():
        return runs_df

    print(
        f"Warning: {int(missing_soil_label.sum()):,} staged runs are missing soil_map_texture_group; "
        "labeling them as AOI/default soil fallback rows."
    )
    runs_df.loc[missing_soil_label, "soil_map_texture_group"] = "Unmapped soil - using AOI defaults"
    if "soil_map_join" in runs_df.columns:
        runs_df.loc[missing_soil_label, "soil_map_join"] = "no_hwsd2_representative_point_match_using_aoi_defaults"
    return runs_df


runs = fill_missing_soil_map_labels(runs)
missing_required_columns = [column for column in required_notebook04_columns if column not in runs.columns]
if REQUIRE_REAL_STAGED_INPUTS and missing_required_columns:
    raise ValueError(
        "The staged SCEPTER run table does not include the lean notebook 04 provenance columns, "
        "and they could not be recovered from scepter_model_units.csv. Rerun notebook 04 first. "
        f"Missing columns: {missing_required_columns}"
    )

if REQUIRE_REAL_STAGED_INPUTS:
    placeholder_rows = runs.astype(str).apply(lambda col: col.str.contains("placeholder", case=False, na=False)).any(axis=1)
    if placeholder_rows.any():
        raise ValueError(
            f"Found {int(placeholder_rows.sum())} staged runs using placeholder inputs. "
            "Rerun notebook 04 with ESA cropland, HWSD2, and CHIRPS artifacts available."
        )

    missing_required_values = runs[required_notebook04_columns].isna().sum()
    missing_required_values = missing_required_values[missing_required_values > 0]
    if not missing_required_values.empty:
        raise ValueError(
            "Notebook 04 provenance columns exist but contain missing values: "
            f"{missing_required_values.to_dict()}"
        )

print(f"Available staged runs: {len(runs):,}")
print("Cropland source:", runs["cropland_source"].dropna().astype(str).iloc[0])
print("Soil map labels:", sorted(runs["soil_map_texture_group"].dropna().astype(str).unique()))
print("Soil source:", runs["soil_source"].dropna().astype(str).iloc[0])
print("Rainfall source:", runs["rainfall_source"].dropna().astype(str).iloc[0])
print("Rainfall months:", runs["rainfall_months_used"].dropna().astype(str).iloc[0])
runs.groupby("scenario_id").size().rename("run_count").reset_index()


Available staged runs: 400
Cropland source: ESA WorldCover cropland mask
Soil map labels: ['Clay', 'Clay loam', 'Loam']
Soil source: /Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/processed/soil/hwsd2/hwsd2_mdb_HWSD2_LAYERS.csv
Rainfall source: CHIRPS rainfall_chirps_monthly via DE Africa S3
Rainfall months: 2021-01,2021-02,2021-03,2021-04,2021-05,2021-06,2021-07,2021-08,2021-09,2021-10,2021-11,2021-12


,scenario_id,run_count
0,baseline_no_erw,100
1,erw_10t_fine,100
2,erw_20t_fine,100
3,erw_40t_medium,100


## Select Runs For This Batch

`smoke` selects one run, `pilot` selects the first four sorted runs, and `final` selects all staged runs. Run `pilot` before `final` so notebook `06` can verify baseline-vs-scenario additionality for one model unit.


In [4]:
selected_runs = select_scepter_runs(
    runs,
    max_runs=MAX_TEST_RUNS,
    scenario_ids=SCENARIO_IDS,
)

print(f"Runs selected for this batch: {len(selected_runs):,}")
preview_columns = [
    "run_id",
    "model_unit_id",
    "scenario_id",
    "area_ha",
    "cropland_source",
    "soil_map_texture_group",
    "basalt_application_t_ha",
    "basalt_d50_um",
]
selected_runs[preview_columns].head(20)


Runs selected for this batch: 400


,run_id,model_unit_id,scenario_id,area_ha,cropland_source,soil_map_texture_group,basalt_application_t_ha,basalt_d50_um
0,mu_00001__baseline_no_erw,mu_00001,baseline_no_erw,1386.31,ESA WorldCover cropland mask,Loam,0.0,50.0
1,mu_00001__erw_10t_fine,mu_00001,erw_10t_fine,1386.31,ESA WorldCover cropland mask,Loam,10.0,50.0
2,mu_00001__erw_20t_fine,mu_00001,erw_20t_fine,1386.31,ESA WorldCover cropland mask,Loam,20.0,50.0
3,mu_00001__erw_40t_medium,mu_00001,erw_40t_medium,1386.31,ESA WorldCover cropland mask,Loam,40.0,100.0
4,mu_00002__baseline_no_erw,mu_00002,baseline_no_erw,1318.87,ESA WorldCover cropland mask,Loam,0.0,50.0
5,mu_00002__erw_10t_fine,mu_00002,erw_10t_fine,1318.87,ESA WorldCover cropland mask,Loam,10.0,50.0
6,mu_00002__erw_20t_fine,mu_00002,erw_20t_fine,1318.87,ESA WorldCover cropland mask,Loam,20.0,50.0
7,mu_00002__erw_40t_medium,mu_00002,erw_40t_medium,1318.87,ESA WorldCover cropland mask,Loam,40.0,100.0
8,mu_00003__baseline_no_erw,mu_00003,baseline_no_erw,1310.58,ESA WorldCover cropland mask,Loam,0.0,50.0
9,mu_00003__erw_10t_fine,mu_00003,erw_10t_fine,1310.58,ESA WorldCover cropland mask,Loam,10.0,50.0


## Stage Run Configs

Each run gets a folder under `data/scepter_runs/runs/` and a JSON configuration file. The config includes site attributes, SCEPTER-ready HWSD2 soil defaults, CHIRPS rainfall/runoff inputs, amendment scenario settings, ESA cropland provenance, HWSD2 soil-map labels, and `input_sources` metadata.


In [5]:
execution_table = write_run_configs(
    selected_runs,
    run_root=RUN_ROOT,
    output_root=OUTPUT_ROOT,
)

execution_table.head()


,run_id,model_unit_id,scenario_id,run_dir,output_dir,config_path,status,input_status,cropland_source,cropland_source_path,cropland_pixels,cropland_pixel_area_m2,cropland_area_note,soil_map_hwsd2_unit_id,soil_map_texture_group,soil_map_wrb4,soil_map_fao90,soil_map_clay_pct,soil_map_sand_pct,soil_map_silt_pct,soil_map_source_path,soil_map_join,soil_source,soil_source_path,soil_note,rainfall_source,rainfall_source_path,rainfall_months_used,runoff_note
0,mu_00001__baseline_no_erw,mu_00001,baseline_no_erw,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/runs/mu_00001__baseline_no_erw,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/outputs/mu_00001__baseline_no_erw,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/runs/mu_00001__baseline_no_erw/scepter_config.json,staged,esa_cropland_block_with_hwsd2_soil_map_chirps_rainfall_runoff_estimate,ESA WorldCover cropland mask,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_cdr_ugagric/data/processed/landuse_landcover/ug_agric_21landuse.tif,138631,100.0,geographic raster; used ESA WorldCover nominal 10 m pixel area,27462,Loam,FLdy,FLd,23,49,28,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/processed/soil/hwsd2/hwsd2_aoi_soil_types.geojson,representative_point_within_hwsd2_aoi_soil_type,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/processed/soil/hwsd2/hwsd2_mdb_HWSD2_LAYERS.csv,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/processed/soil/hwsd2/scepter_soil_defaults_hwsd2.csv,AOI pixel-share weighted HWSD2 defaults for first-pass SCEPTER staging,CHIRPS rainfall_chirps_monthly via DE Africa S3,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/processed/climate/rainfall/monthly_rainfall_aoi_jan_jun_2026.csv,"2021-01,2021-02,2021-03,2021-04,2021-05,2021-06,2021-07,2021-08,2021-09,2021-10,2021-11,2021-12",runoff_mm_yr estimated as 0.25 * precipitation_mm_yr
1,mu_00001__erw_10t_fine,mu_00001,erw_10t_fine,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/runs/mu_00001__erw_10t_fine,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/outputs/mu_00001__erw_10t_fine,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/runs/mu_00001__erw_10t_fine/scepter_config.json,staged,esa_cropland_block_with_hwsd2_soil_map_chirps_rainfall_runoff_estimate,ESA WorldCover cropland mask,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_cdr_ugagric/data/processed/landuse_landcover/ug_agric_21landuse.tif,138631,100.0,geographic raster; used ESA WorldCover nominal 10 m pixel area,27462,Loam,FLdy,FLd,23,49,28,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/processed/soil/hwsd2/hwsd2_aoi_soil_types.geojson,representative_point_within_hwsd2_aoi_soil_type,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/processed/soil/hwsd2/hwsd2_mdb_HWSD2_LAYERS.csv,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/processed/soil/hwsd2/scepter_soil_defaults_hwsd2.csv,AOI pixel-share weighted HWSD2 defaults for first-pass SCEPTER staging,CHIRPS rainfall_chirps_monthly via DE Africa S3,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/processed/climate/rainfall/monthly_rainfall_aoi_jan_jun_2026.csv,"2021-01,2021-02,2021-03,2021-04,2021-05,2021-06,2021-07,2021-08,2021-09,2021-10,2021-11,2021-12",runoff_mm_yr estimated as 0.25 * precipitation_mm_yr
2,mu_00001__erw_20t_fine,mu_00001,erw_20t_fine,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/runs/mu_00001__erw_20t_fine,/Users/paullwanga/P

## Inspect One Config

Before running an external model, inspect one generated config to confirm raster-block area, ESA cropland source, soil-map label, rainfall inputs, and scenario parameters look reasonable.


In [6]:
example_config = Path(execution_table.iloc[0]["config_path"])
print(example_config)
print(example_config.read_text()[:2000])


/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/runs/mu_00001__baseline_no_erw/scepter_config.json
{
  "run_id": "mu_00001__baseline_no_erw",
  "model_unit_id": "mu_00001",
  "scenario_id": "baseline_no_erw",
  "site": {
    "area_ha": 1386.31,
    "centroid_lon": 31.56975,
    "centroid_lat": 1.0858333333333337
  },
  "soil": {
    "ph": 6.2,
    "cec_cmolc_kg": 14.0,
    "clay_pct": 27.87395961118477,
    "bulk_density_g_cm3": 1.3,
    "depth_cm": 30.0
  },
  "climate": {
    "temperature_c": 24.0,
    "precipitation_mm_yr": 1132.2522133103907,
    "runoff_mm_yr": 283.0630533275977
  },
  "amendment": {
    "basalt_application_t_ha": 0.0,
    "basalt_d50_um": 50.0
  },
  "simulation": {
    "years": 10
  },
  "outputs": {
    "output_dir": "/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/outputs/mu_00001__baseline_no_erw",
    "summary_csv": "/Users/paullwanga/Programming_projects/enha

## Execute Or Stage Manifest

When `RUN_EXTERNAL_SCEPTER = False`, this cell writes a manifest showing that ESA/HWSD2/CHIRPS-backed configs were staged but not executed. When set to `True`, it calls the command template once per run.

The command template can use these fields: `{run_id}`, `{run_dir}`, `{output_dir}`, `{config_path}`, `{model_unit_id}`, and `{scenario_id}`.


In [7]:
if RUN_EXTERNAL_SCEPTER:
    scepter_executable = SCEPTER_COMMAND_TEMPLATE.split()[0]
    if shutil.which(scepter_executable) is None:
        raise RuntimeError(
            f"RUN_EXTERNAL_SCEPTER is True, but the command `{scepter_executable}` is not available in this kernel. "
            "Install SCEPTER in the active Colab/Jupyter runtime, or update SCEPTER_COMMAND_TEMPLATE "
            "to the full executable/script path. If you only want to stage configs for now, set "
            "RUN_EXTERNAL_SCEPTER = False."
        )

    if MANIFEST_PATH.exists():
        previous_manifest = pd.read_csv(MANIFEST_PATH)
        completed_run_ids = set(previous_manifest.loc[previous_manifest["status"] == "complete", "run_id"])
    else:
        previous_manifest = pd.DataFrame()
        completed_run_ids = set()

    completed_summary_ids = {
        row["run_id"]
        for row in execution_table.to_dict(orient="records")
        if (Path(row["output_dir"]) / f"{row['run_id']}_summary.csv").exists()
    }
    completed_run_ids = completed_run_ids.union(completed_summary_ids)
    pending_execution_table = execution_table[~execution_table["run_id"].isin(completed_run_ids)].copy()
    print(f"Resuming SCEPTER execution: {len(completed_run_ids):,} complete, {len(pending_execution_table):,} pending.")

    if pending_execution_table.empty:
        manifest = previous_manifest.copy()
    else:
        new_manifest = execute_scepter_runs(
            pending_execution_table,
            command_template=SCEPTER_COMMAND_TEMPLATE,
            timeout_seconds=TIMEOUT_SECONDS,
        )
        if previous_manifest.empty:
            manifest = new_manifest
        else:
            previous_keep = previous_manifest[previous_manifest["run_id"].isin(completed_run_ids)].copy()
            manifest = pd.concat([previous_keep, new_manifest], ignore_index=True)
else:
    manifest = execution_table.copy()
    manifest["command"] = SCEPTER_COMMAND_TEMPLATE
    manifest["status"] = "staged_not_executed"
    manifest["return_code"] = pd.NA
    manifest["elapsed_seconds"] = 0.0
    manifest["stdout_log"] = pd.NA
    manifest["stderr_log"] = pd.NA

manifest = manifest.sort_values(["scenario_id", "run_id"]).reset_index(drop=True)
manifest.to_csv(MANIFEST_PATH, index=False)
manifest[["run_id", "scenario_id", "status", "config_path", "output_dir"]].head(20)


,run_id,scenario_id,status,config_path,output_dir
0,mu_00001__baseline_no_erw,baseline_no_erw,staged_not_executed,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/runs/mu_00001__baseline_no_erw/scepter_config.json,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/outputs/mu_00001__baseline_no_erw
1,mu_00002__baseline_no_erw,baseline_no_erw,staged_not_executed,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/runs/mu_00002__baseline_no_erw/scepter_config.json,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/outputs/mu_00002__baseline_no_erw
2,mu_00003__baseline_no_erw,baseline_no_erw,staged_not_executed,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/runs/mu_00003__baseline_no_erw/scepter_config.json,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/outputs/mu_00003__baseline_no_erw
3,mu_00004__baseline_no_erw,baseline_no_erw,staged_not_executed,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/runs/mu_00004__baseline_no_erw/scepter_config.json,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/outputs/mu_00004__baseline_no_erw
4,mu_00005__baseline_no_erw,baseline_no_erw,staged_not_executed,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/runs/mu_00005__baseline_no_erw/scepter_config.json,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/outputs/mu_00005__baseline_no_erw
5,mu_00006__baseline_no_erw,baseline_no_erw,staged_not_executed,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/runs/mu_00006__baseline_no_erw/scepter_config.json,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/outputs/mu_00006__baseline_no_erw
6,mu_00007__baseline_no_erw,baseline_no_erw,staged_not_executed,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/runs/mu_00007__baseline_no_erw/scepter_config.json,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/outputs/mu_00007__baseline_no_erw
7,mu_00008__baseline_no_erw,baseline_no_erw,staged_not_executed,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/runs/mu_00008__baseline_no_erw/scepter_config.json,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/outputs/mu_00008__baseline_no_erw
8,mu_00009__baseline_no_erw,baseline_no_erw,staged_not_executed,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/runs/mu_00009__baseline_no_erw/scepter_config.json,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/outputs/mu_00009__baseline_no_erw
9,mu_00010__baseline_no_erw,baseline_no_erw,staged_not_executed,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/runs/mu_00010__baseline_no_erw/scepter_config.json,/Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_spatial_mrv/data/scepter_runs/outputs/mu_00010__baseline_no_erw


## Batch Summary


In [8]:
status_summary = manifest.groupby("status").size().rename("run_count").reset_index()
scenario_summary = manifest.groupby(["scenario_id", "status"]).size().rename("run_count").reset_index()
source_columns = [
    column
    for column in [
        "cropland_source",
        "cropland_source_path",
        "soil_map_texture_group",
        "soil_map_source_path",
        "soil_source",
        "rainfall_source",
        "rainfall_months_used",
        "missing_requested_months",
    ]
    if column in manifest.columns
]
source_summary = manifest[source_columns].drop_duplicates().reset_index(drop=True) if source_columns else pd.DataFrame()

status_summary, scenario_summary, source_summary, MANIFEST_PATH


(                status  run_count
 0  staged_not_executed        400,
        scenario_id               status  run_count
 0  baseline_no_erw  staged_not_executed        100
 1     erw_10t_fine  staged_not_executed        100
 2     erw_20t_fine  staged_not_executed        100
 3   erw_40t_medium  staged_not_executed        100,
                 cropland_source  \
 0  ESA WorldCover cropland mask   
 1  ESA WorldCover cropland mask   
 2  ESA WorldCover cropland mask   
 
                                                                                                                       cropland_source_path  \
 0  /Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_cdr_ugagric/data/processed/landuse_landcover/ug_agric_21landuse.tif   
 1  /Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_cdr_ugagric/data/processed/landuse_landcover/ug_agric_21landuse.tif   
 2  /Users/paullwanga/Programming_projects/enhanced_rock_weathering/erw_cdr_ugagric/data/proc

## Expected SCEPTER Outputs

Once external execution is enabled, each run folder under `data/scepter_runs/outputs/{run_id}/` should contain whatever files SCEPTER produces for that ESA-cropland/HWSD2/CHIRPS-backed scenario. The next notebook parses those outputs into a clean table with at least:

- `run_id`
- `model_unit_id`
- `scenario_id`
- weathering or alkalinity metric
- cumulative CO2 removal
- CO2 removal per hectare
- soil pH or chemistry changes if available

The parser will use `scepter_execution_manifest.csv` to know which runs completed and where their output folders are.


## Outputs From This Notebook

This notebook writes SCEPTER batch execution artifacts:

- `data/scepter_runs/runs/{run_id}/scepter_config.json`: one model config per selected run.
- `data/scepter_runs/outputs/{run_id}/`: output folder reserved for each selected run.
- `data/scepter_runs/outputs/scepter_execution_manifest.csv`: status table for staged or executed runs.
- `data/scepter_runs/runs/{run_id}/scepter_stdout.log`: stdout log, written only when `RUN_EXTERNAL_SCEPTER = True`.
- `data/scepter_runs/runs/{run_id}/scepter_stderr.log`: stderr log, written only when `RUN_EXTERNAL_SCEPTER = True`.

The next step is notebook `06`, which should read the execution manifest, parse completed SCEPTER output files, and join model results back to raster-derived model units.
